# 01 - ReAct Agent

ReAct (Reasoning + Acting) agent implementation.


In [ ]:
import numpy as np
import re
import math
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. ReAct Framework Overview


In [ ]:
# ReAct: Thought -> Action -> Observation -> Answer
print('ReAct Pattern: Thought -> Action -> Observation -> Answer')


## 2. Tool Implementations


In [ ]:
KNOWLEDGE_BASE = {
    'france population': '67 million',
    'germany population': '83 million',
    'capital of france': 'Paris',
    'capital of germany': 'Berlin',
    'area of france': '551,695 sq km',
    'area of germany': '357,022 sq km',
    'eiffel tower height': '330 meters',
    'mount everest height': '8,849 meters'
}

def search_tool(query):
    query = query.lower().strip()
    if query in KNOWLEDGE_BASE:
        return KNOWLEDGE_BASE[query]
    for key, value in KNOWLEDGE_BASE.items():
        if all(word in key for word in query.split()):
            return value
    return 'No information found'

def calculator_tool(expression):
    try:
        expr = re.sub(r'[^0-9+\-*/().sqrtlogexp ]', '', expression)
        allowed = {'sqrt': math.sqrt, 'log': math.log, 'exp': math.exp, 'sin': math.sin, 'cos': math.cos, 'pi': math.pi, 'e': math.e}
        result = eval(expr, {'__builtins__': {}}, allowed)
        return str(round(result, 4))
    except Exception as e:
        return f'Error: {str(e)}'

def lookup_tool(entity):
    return search_tool(entity)

TOOLS = {'search': search_tool, 'calculator': calculator_tool, 'lookup': lookup_tool}
print(f'Tools ready: {list(TOOLS.keys())}')


## 3. ReAct Agent Implementation


In [ ]:
class ReActAgent:
    def __init__(self, tools, max_steps=10):
        self.tools = tools
        self.max_steps = max_steps
        self.trace = []
    def parse_action(self, text):
        match = re.search(r'(\w+)\[(.*?)\]', text)
        if match:
            return match.group(1), match.group(2)
        return None, None
    def think(self, query, observations):
        if not observations:
            if any(op in query for op in ['+', '-', '*', '/', 'sqrt', 'log']):
                return f'Thought: This is a calculation problem.\nAction: calculator[{query}]'
            else:
                return f'Thought: I need to search for {query}.\nAction: search[{query}]'
        else:
            last_obs = observations[-1]
            if 'Error' in last_obs or 'No information' in last_obs:
                return f'Thought: Previous action failed. Trying lookup.\nAction: lookup[{query}]'
            else:
                return f'Thought: I have enough information.\nAction: finish[{last_obs}]'
    def run(self, query):
        self.trace = []
        observations = []
        print(f'ReAct Agent: {query}')
        for step in range(self.max_steps):
            thought_action = self.think(query, observations)
            print(f'\nStep {step+1}:')
            print(thought_action)
            self.trace.append(thought_action)
            action_name, action_arg = self.parse_action(thought_action)
            if action_name == 'finish':
                print(f'Final Answer: {action_arg}')
                return action_arg
            if action_name in self.tools:
                observation = self.tools[action_name](action_arg)
                print(f'Observation: {observation}')
                observations.append(observation)
                self.trace.append(f'Observation: {observation}')
            else:
                print(f'Unknown action: {action_name}')
                observations.append('Error: Unknown action')
        return 'Max steps reached'

agent = ReActAgent(TOOLS)
print('ReAct Agent initialized!')


In [ ]:
test_queries = ['What is the population of France?', 'Calculate sqrt(144) * 2', 'What is the capital of Germany?']
for q in test_queries:
    result = agent.run(q)
    print(f'Result: {result}\n')


## 4. Multi-Step Problem Solving


In [ ]:
complex_query = 'If France has 67 million people and Germany has 83 million, what is the total?'

class AdvancedReActAgent(ReActAgent):
    def think(self, query, observations):
        if not observations:
            if 'france' in query.lower() and 'germany' in query.lower():
                return 'Thought: Need both populations.\nAction: search[France population]'
            return super().think(query, observations)
        last_obs = observations[-1] if observations else ''
        if 'france' in query.lower() and len(observations) == 1 and '67' in last_obs:
            return 'Thought: Found France. Now Germany.\nAction: search[Germany population]'
        if len(observations) == 2 and '83' in last_obs:
            return 'Thought: Both found. Adding.\nAction: calculator[67 + 83]'
        if len(observations) == 3 and any(c.isdigit() for c in last_obs):
            return f'Thought: Calculated total.\nAction: finish[{last_obs} million]'
        return super().think(query, observations)

adv_agent = AdvancedReActAgent(TOOLS)
result = adv_agent.run(complex_query)
print(f'Multi-step result: {result}')


## 5. Trace Visualization


In [ ]:
def visualize_trace(trace):
    thoughts = [t for t in trace if t.startswith('Thought')]
    observations = [t for t in trace if t.startswith('Observation')]
    fig, ax = plt.subplots(figsize=(12, len(thoughts) * 0.8 + 1))
    ax.axis('off')
    y_pos = len(thoughts)
    for i, thought in enumerate(thoughts):
        ax.text(0.05, y_pos - i, f'Step {i+1}:', fontsize=12, fontweight='bold')
        ax.text(0.1, y_pos - i - 0.3, thought.replace('Thought: ', '💭 '), fontsize=10, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
        if i < len(observations):
            ax.text(0.1, y_pos - i - 0.7, observations[i].replace('Observation: ', '👁 '), fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
    plt.title('ReAct Agent Trace', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

agent2 = ReActAgent(TOOLS)
agent2.run('What is the Eiffel Tower height?')
visualize_trace(agent2.trace)


## 6. Exercises


In [ ]:
# Exercise 1: Add reasoning tool
# Exercise 2: Implement backtracking
# Exercise 3: Add memory
# Exercise 4: File reading agent
print('Exercises listed!')
